# GTEx top-gene loading plots for heart and liver LVs

1. Load the top SHAP-ranked LV for Heart - Atrial Appendage, Heart - Left Ventricle, and Liver
2. Pull the top 1% genes by loading from the GTEx CLAMP Z matrix
3. Plot the loading tail and label the top 10 genes for each tissue
4. Save panel-ready CSVs and figure files


💡 **Environment:** `clamp-analyses`

## Libraries


In [ ]:
library(here)
library(dplyr)
library(ggplot2)
library(ggrepel)

## Settings


In [ ]:
TARGET_TISSUES <- c("Heart - Atrial Appendage", "Heart - Left Ventricle", "Liver")
EXTRA_LVS     <- c("LV5", "LV293")
TOP_GENE_PCT  <- 0.01
N_LABEL_GENES <- 10
MODEL_PATH    <- here("output/01_model_building/02_gtex/01_CLAMP/CLAMPfull.rds")
SHAP_PATH     <- here("output/03_model_biology/01_gtex/02_LV_importance_kmeans_biology/all_shap_positive.tsv")
OUT_DIR       <- here("output/99_panels/fig2")

set.seed(42)

## Top LV per tissue


In [ ]:
sanitize_stub <- function(x) {
    x <- tolower(x)
    x <- gsub("[^[:alnum:]]+", "_", x)
    x <- gsub("^_|_$", "", x)
    gsub("_+", "_", x)
}

shap_df <- read.delim(SHAP_PATH, sep = "\t", stringsAsFactors = FALSE)

top_lv_tbl <- shap_df %>%
    filter(Tissue %in% TARGET_TISSUES) %>%
    group_by(Tissue) %>%
    arrange(Rank, .by_group = TRUE) %>%
    slice_head(n = 1) %>%
    ungroup() %>%
    mutate(
        Tissue_chr = Tissue,
        panel     = paste0(Tissue_chr, "\n", Feature),
        file_stub = paste(sanitize_stub(Tissue_chr), sanitize_stub(Feature), sep = "_"),
        Tissue    = factor(Tissue_chr, levels = TARGET_TISSUES)
    ) %>%
    select(-Tissue_chr)

stopifnot(nrow(top_lv_tbl) == length(TARGET_TISSUES))

top_lv_tbl %>%
    select(Tissue, Feature, Mean_SHAP_Tissue, Rank, file_stub)

## Append extra LVs (LV5, LV293)

In [ ]:
extra_lv_tbl <- data.frame(
    Tissue           = factor(EXTRA_LVS, levels = EXTRA_LVS),
    Feature          = EXTRA_LVS,
    Mean_SHAP_Tissue = NA_real_,
    Rank             = NA_integer_,
    panel            = EXTRA_LVS,
    file_stub        = tolower(gsub("LV", "lv", EXTRA_LVS)),
    stringsAsFactors = FALSE
)

top_lv_tbl <- bind_rows(top_lv_tbl, extra_lv_tbl)
top_lv_tbl$Tissue <- factor(top_lv_tbl$Tissue, levels = as.character(top_lv_tbl$Tissue))

top_lv_tbl %>% select(Tissue, Feature, Mean_SHAP_Tissue, Rank, file_stub)

## Top 1% genes by loading


In [ ]:
clamp_model <- readRDS(MODEL_PATH)
Z <- as.matrix(clamp_model$Z)
rm(clamp_model); gc()

cat("Z matrix:", nrow(Z), "genes x", ncol(Z), "LVs\n")

get_top_loading_genes <- function(tissue, lv, file_stub, z_mat, top_gene_pct = 0.01, n_label_genes = 10) {
    loadings <- z_mat[, lv]
    n_top    <- max(1L, ceiling(length(loadings) * top_gene_pct))
    ord      <- order(loadings, decreasing = TRUE)

    data.frame(
        Tissue  = tissue,
        LV      = lv,
        panel   = paste0(tissue, "\n", lv),
        file_stub = file_stub,
        rank    = seq_len(n_top),
        gene    = rownames(z_mat)[ord][seq_len(n_top)],
        loading = loadings[ord][seq_len(n_top)],
        label   = ifelse(seq_len(n_top) <= n_label_genes,
                         rownames(z_mat)[ord][seq_len(n_top)],
                         NA_character_),
        stringsAsFactors = FALSE
    )
}

top_gene_loadings <- bind_rows(lapply(seq_len(nrow(top_lv_tbl)), function(i) {
    get_top_loading_genes(
        tissue        = as.character(top_lv_tbl$Tissue[i]),
        lv            = top_lv_tbl$Feature[i],
        file_stub     = top_lv_tbl$file_stub[i],
        z_mat         = Z,
        top_gene_pct  = TOP_GENE_PCT,
        n_label_genes = N_LABEL_GENES
    )
}))

rm(Z); gc()

top_gene_loadings$panel <- factor(top_gene_loadings$panel, levels = top_lv_tbl$panel)
top_label_genes <- top_gene_loadings %>% filter(!is.na(label))

top_gene_loadings %>%
    group_by(Tissue, LV) %>%
    slice_head(n = 12) %>%
    ungroup()

## Plot top-1% loading tails


In [ ]:
theme_loading_plot <- function(base_size = 11) {
    theme_classic(base_size = base_size) %+replace%
        theme(
            axis.line    = element_line(color = "black", linewidth = 0.4),
            axis.ticks   = element_line(color = "black", linewidth = 0.35),
            axis.text.x  = element_blank(),
            axis.ticks.x = element_blank(),
            axis.title.x = element_blank(),
            plot.title   = element_text(face = "bold", hjust = 0.5, size = base_size + 1),
            plot.margin  = margin(5.5, 28, 5.5, 5.5)
        )
}

make_loading_plot <- function(plot_df, label_df, title_text) {
    ggplot(plot_df, aes(x = rank, y = loading)) +
        geom_point(size = 1.2, color = "grey75") +
        geom_point(
            data = label_df,
            color = "#C23B22",
            size = 1.6
        ) +
        geom_text_repel(
            data = label_df,
            aes(label = label),
            size = 3,
            fontface = "italic",
            direction = "y",
            hjust = 0,
            seed = 42,
            max.overlaps = Inf,
            min.segment.length = 0,
            box.padding = 0.3,
            point.padding = 0.2,
            nudge_x = 8,
            segment.color = "grey60",
            segment.size = 0.25
        ) +
        coord_cartesian(clip = "off") +
        scale_x_continuous(expand = expansion(mult = c(0.02, 0.18))) +
        labs(title = title_text, y = "Loadings") +
        theme_loading_plot()
}

lv_plots <- setNames(lapply(seq_len(nrow(top_lv_tbl)), function(i) {
    plot_df <- top_gene_loadings %>% filter(file_stub == top_lv_tbl$file_stub[i])
    label_df <- top_label_genes %>% filter(file_stub == top_lv_tbl$file_stub[i])

    make_loading_plot(
        plot_df = plot_df,
        label_df = label_df,
        title_text = as.character(top_lv_tbl$panel[i])
    )
}), top_lv_tbl$file_stub)

options(repr.plot.width = 5.2, repr.plot.height = 4.8)
invisible(lapply(lv_plots, print))

## Save panel-ready outputs


In [ ]:
dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

write.csv(top_lv_tbl, file.path(OUT_DIR, "top_lv_selection.csv"), row.names = FALSE)
write.csv(top_gene_loadings, file.path(OUT_DIR, "top_lv_gene_loadings.csv"), row.names = FALSE)
write.csv(top_label_genes, file.path(OUT_DIR, "top_lv_gene_labels.csv"), row.names = FALSE)

unlink(file.path(OUT_DIR, c("top_lv_gene_loadings.png", "top_lv_gene_loadings.pdf")), force = TRUE)

for (i in seq_len(nrow(top_lv_tbl))) {
    stub <- top_lv_tbl$file_stub[i]
    p <- lv_plots[[stub]]

    ggsave(
        filename = file.path(OUT_DIR, paste0("top_lv_gene_loadings_", stub, ".png")),
        plot = p,
        width = 5,
        height = 4,
        dpi = 300,
        bg = "white"
    )

    ggsave(
        filename = file.path(OUT_DIR, paste0("top_lv_gene_loadings_", stub, ".pdf")),
        plot = p,
        width = 5,
        height = 4,
        bg = "white"
    )
}

cat("Saved files to:", OUT_DIR, "\n")
